In [10]:
import re
from pathlib import Path

import numpy as np
import pandas as pd
import statsmodels.api as sm
from sklearn.preprocessing import StandardScaler

from run1.lib.classes_ml import DataHandler

In [11]:
BASE_DIR = Path.cwd()  # Current directory of the running file
ROOT_DIR = BASE_DIR.parent.parent
DATA_DIR = ROOT_DIR / "run1" / "data"
CURRENT_DIR = BASE_DIR

In [12]:
_df = pd.read_excel(DATA_DIR / "S02_data_exp.xlsx")
print(f"df.shape: {_df.shape}")

df.shape: (378, 9)


In [13]:
_df

,sample_no,location,position,R,W,D,stress_value_5052,stress_value_6061,stress_value_center
0,1,1,0.153846,1400,60,10,28.0,51.0,12.0
1,2,1,0.153846,1400,60,15,14.0,-21.0,17.0
2,3,1,0.153846,1400,60,20,10.0,35.0,12.0
3,4,1,0.153846,1400,70,10,10.0,-10.0,20.0
4,5,1,0.153846,1400,70,15,6.0,41.0,14.0
...,...,...,...,...,...,...,...,...,...
373,50,7,0.846154,1600,70,15,4.0,-23.0,2.0
374,51,7,0.846154,1600,70,20,0.0,-1.0,2.0
375,52,7,0.846154,1600,80,10,-2.0,-41.0,5.0
376,53,7,0.846154,1600,80,15,10.0,-90.0,1.0


In [14]:
# Load location info
locs = pd.read_excel(DATA_DIR / "S04_loc_values.xlsx").rename(
    columns={
        "Location": "location",
        "Fx": "Fx_location",
        "Fy": "Fy_location",
        "Fz": "Fz_location",
        "Mz": "Mz_location",
    }
)
locs = locs.drop(columns=["loc_idx", "loc_time"])
locs

,sample_no,location,Fx_location,Fy_location,Fz_location,Mz_location
0,1,1,-0.077671,0.143026,1.244326,1.873865
1,1,2,-0.027450,0.207481,1.625039,4.615337
2,1,3,-0.038704,0.272508,1.799653,5.209334
3,1,4,-0.023484,0.278517,1.882699,5.764727
4,1,5,-0.038251,0.280276,1.955412,6.097497
...,...,...,...,...,...,...
373,54,3,-0.079764,0.196450,1.352393,5.512565
374,54,4,-0.091477,0.259717,1.529089,6.898179
375,54,5,-0.068128,0.241906,1.657737,7.940597
376,54,6,-0.066880,0.261006,1.712860,8.849661


In [15]:
_df = _df.merge(locs, on=["sample_no", "location"])
_df

,sample_no,location,position,R,W,D,stress_value_5052,stress_value_6061,stress_value_center,Fx_location,Fy_location,Fz_location,Mz_location
0,1,1,0.153846,1400,60,10,28.0,51.0,12.0,-0.077671,0.143026,1.244326,1.873865
1,2,1,0.153846,1400,60,15,14.0,-21.0,17.0,-0.133276,0.164254,1.203367,-1.054677
2,3,1,0.153846,1400,60,20,10.0,35.0,12.0,-0.059639,0.269418,1.444542,2.940728
3,4,1,0.153846,1400,70,10,10.0,-10.0,20.0,-0.051020,0.211907,1.601667,3.661974
4,5,1,0.153846,1400,70,15,6.0,41.0,14.0,-0.100744,0.179582,1.095031,-0.016799
...,...,...,...,...,...,...,...,...,...,...,...,...,...
373,50,7,0.846154,1600,70,15,4.0,-23.0,2.0,-0.084714,0.284958,1.905742,8.950274
374,51,7,0.846154,1600,70,20,0.0,-1.0,2.0,-0.094956,0.257101,1.669120,9.627879
375,52,7,0.846154,1600,80,10,-2.0,-41.0,5.0,-0.203323,0.173404,1.671576,4.696642
376,53,7,0.846154,1600,80,15,10.0,-90.0,1.0,-0.099644,0.266207,1.686495,10.497974


In [16]:
# Select columns for features and targets
colsY = [c for c in _df.columns if re.search(r"stress_value", c)]

# Select feature columns based on predefined names
colsY = [c for c in colsY if c in ["stress_value_center"]]

# Predefined feature columns
# colsX = [c for c in _df.columns if c in ["R", "W", "D", "position", "Fx_location", "Fy_location", "Fz_location", "Mz_location"]]
colsX = [c for c in _df.columns if c in ["R", "W", "D", "position"]]
_dfY = _df[colsY]
_dfX = _df[colsX]
print("Selected feature columns:", colsX)
print("Selected target columns:", colsY)
print(f"dfX.shape: {_dfX.shape}")
print(f"dfY.shape: {_dfY.shape}")

# %% Extract features and targets
_X = _dfX.values
_Y = _dfY.values
print(f"_X.shape: {_X.shape}")
print(f"_Y.shape: {_Y.shape}")

# Create DataHandler instance
data_handler = DataHandler(
    _X=_X,
    _Y=_Y,
    scalerX=StandardScaler(),
    scalerY=StandardScaler(),
    colsX=colsX,
    colsY=colsY,
)
idx = 1
random_state = 1
test_size = 0.0
data_handler.split_and_scale(random_state=random_state, test_size=test_size)
df_X_train, df_Y_train = data_handler.get_train(as_dataframe=True)
display(df_X_train.head())
display(df_Y_train.head())

# Analyze model summary (all predictors)
X = sm.add_constant(df_X_train)
model = sm.OLS(df_Y_train, X).fit()

# Get and print model summary
model_summary = model.summary()
print(model_summary)

# Get p-values and sort features by significance
df_table = model.summary2().tables[1]
df_table = df_table.sort_values(by="P>|t|", ascending=True)
display(df_table)

ranking = df_table.reset_index()[["index", "P>|t|"]].rename(
    columns={"index": "feature", "P>|t|": "value"}
)
ranking["measure"] = "OLS_p_value"
ranking["rank"] = np.arange(1, len(ranking) + 1)
# Remove constant term from ranking
ranking = ranking[ranking["feature"] != "const"]
display(ranking)

Selected feature columns: ['position', 'R', 'W', 'D']
Selected target columns: ['stress_value_center']
dfX.shape: (378, 4)
dfY.shape: (378, 1)
_X.shape: (378, 4)
_Y.shape: (378, 1)
No test set, using all data for training.


,position,R,W,D
0,-5.000000e-01,-1.224745,0.000000,-1.224745
1,-4.810966e-16,1.224745,1.224745,0.000000
2,-1.000000e+00,0.000000,-1.224745,-1.224745
3,-5.000000e-01,1.224745,1.224745,1.224745
4,-1.000000e+00,1.224745,0.000000,-1.224745


,stress_value_center
0,-1.088213
1,0.678466
2,-0.499320
3,-0.057650
4,-0.646543


                             OLS Regression Results                            
Dep. Variable:     stress_value_center   R-squared:                       0.290
Model:                             OLS   Adj. R-squared:                  0.282
Method:                  Least Squares   F-statistic:                     38.01
Date:                 Thu, 26 Feb 2026   Prob (F-statistic):           1.11e-26
Time:                         07:50:44   Log-Likelihood:                -471.74
No. Observations:                  378   AIC:                             953.5
Df Residuals:                      373   BIC:                             973.1
Df Model:                            4                                         
Covariance Type:             nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const      -7.546e-17      0.044  -1.73e-1

,Coef.,Std.Err.,t,P>|t|,[0.025,0.975]
position,-4.963548e-01,0.043641,-1.137346e+01,6.197750e-26,-0.582169,-0.410541
R,-1.736327e-01,0.043641,-3.978615e+00,8.329439e-05,-0.259447,-0.087818
W,1.011177e-01,0.043641,2.317009e+00,2.104416e-02,0.015304,0.186932
D,5.338944e-02,0.043641,1.223364e+00,2.219644e-01,-0.032425,0.139204
const,-7.546047e-17,0.043641,-1.729099e-15,1.000000e+00,-0.085814,0.085814


,feature,value,measure,rank
0,position,6.197750e-26,OLS_p_value,1
1,R,8.329439e-05,OLS_p_value,2
2,W,2.104416e-02,OLS_p_value,3
3,D,2.219644e-01,OLS_p_value,4


In [17]:
# Select columns for features and targets
colsY = [c for c in _df.columns if re.search(r"stress_value", c)]

# Select feature columns based on predefined names
colsY = [c for c in colsY if c in ["stress_value_5052"]]

# Predefined feature columns
# colsX = [c for c in _df.columns if c in ["R", "W", "D", "position", "Fx_location", "Fy_location", "Fz_location", "Mz_location"]]
colsX = [c for c in _df.columns if c in ["R", "W", "D", "position"]]
_dfY = _df[colsY]
_dfX = _df[colsX]
print("Selected feature columns:", colsX)
print("Selected target columns:", colsY)
print(f"dfX.shape: {_dfX.shape}")
print(f"dfY.shape: {_dfY.shape}")

# %% Extract features and targets
_X = _dfX.values
_Y = _dfY.values
print(f"_X.shape: {_X.shape}")
print(f"_Y.shape: {_Y.shape}")

# Create DataHandler instance
data_handler = DataHandler(
    _X=_X,
    _Y=_Y,
    scalerX=StandardScaler(),
    scalerY=StandardScaler(),
    colsX=colsX,
    colsY=colsY,
)
idx = 1
random_state = 1
test_size = 0.0
data_handler.split_and_scale(random_state=random_state, test_size=test_size)
df_X_train, df_Y_train = data_handler.get_train(as_dataframe=True)
display(df_X_train.head())
display(df_Y_train.head())

# Analyze model summary (all predictors)
X = sm.add_constant(df_X_train)
model = sm.OLS(df_Y_train, X).fit()

# Get and print model summary
model_summary = model.summary()
print(model_summary)

# Get p-values and sort features by significance
df_table = model.summary2().tables[1]
df_table = df_table.sort_values(by="P>|t|", ascending=True)
display(df_table)

ranking = df_table.reset_index()[["index", "P>|t|"]].rename(
    columns={"index": "feature", "P>|t|": "value"}
)
ranking["measure"] = "OLS_p_value"
ranking["rank"] = np.arange(1, len(ranking) + 1)
# Remove constant term from ranking
ranking = ranking[ranking["feature"] != "const"]
display(ranking)

Selected feature columns: ['position', 'R', 'W', 'D']
Selected target columns: ['stress_value_5052']
dfX.shape: (378, 4)
dfY.shape: (378, 1)
_X.shape: (378, 4)
_Y.shape: (378, 1)
No test set, using all data for training.


,position,R,W,D
0,-5.000000e-01,-1.224745,0.000000,-1.224745
1,-4.810966e-16,1.224745,1.224745,0.000000
2,-1.000000e+00,0.000000,-1.224745,-1.224745
3,-5.000000e-01,1.224745,1.224745,1.224745
4,-1.000000e+00,1.224745,0.000000,-1.224745


,stress_value_5052
0,0.884327
1,0.376763
2,-2.351395
3,-0.955593
4,1.518782


                            OLS Regression Results                            
Dep. Variable:      stress_value_5052   R-squared:                       0.070
Model:                            OLS   Adj. R-squared:                  0.060
Method:                 Least Squares   F-statistic:                     7.026
Date:                Thu, 26 Feb 2026   Prob (F-statistic):           1.84e-05
Time:                        07:50:45   Log-Likelihood:                -522.63
No. Observations:                 378   AIC:                             1055.
Df Residuals:                     373   BIC:                             1075.
Df Model:                           4                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const       2.033e-17      0.050   4.07e-16      1.0

,Coef.,Std.Err.,t,P>|t|,[0.025,0.975]
position,-1.922267e-01,0.049931,-3.849843e+00,0.000139,-0.290408,-0.094045
R,1.654772e-01,0.049931,3.314114e+00,0.001009,0.067296,0.263659
D,-6.888023e-02,0.049931,-1.379507e+00,0.168565,-0.167062,0.029301
W,3.149019e-02,0.049931,6.306735e-01,0.528640,-0.066691,0.129672
const,2.032879e-17,0.049931,4.071372e-16,1.000000,-0.098182,0.098182


,feature,value,measure,rank
0,position,0.000139,OLS_p_value,1
1,R,0.001009,OLS_p_value,2
2,D,0.168565,OLS_p_value,3
3,W,0.528640,OLS_p_value,4


In [18]:
# Select columns for features and targets
colsY = [c for c in _df.columns if re.search(r"stress_value", c)]

# Select feature columns based on predefined names
colsY = [c for c in colsY if c in ["stress_value_6061"]]

# Predefined feature columns
# colsX = [c for c in _df.columns if c in ["R", "W", "D", "position", "Fx_location", "Fy_location", "Fz_location", "Mz_location"]]
colsX = [c for c in _df.columns if c in ["R", "W", "D", "position"]]
_dfY = _df[colsY]
_dfX = _df[colsX]
print("Selected feature columns:", colsX)
print("Selected target columns:", colsY)
print(f"dfX.shape: {_dfX.shape}")
print(f"dfY.shape: {_dfY.shape}")

# %% Extract features and targets
_X = _dfX.values
_Y = _dfY.values
print(f"_X.shape: {_X.shape}")
print(f"_Y.shape: {_Y.shape}")

# Create DataHandler instance
data_handler = DataHandler(
    _X=_X,
    _Y=_Y,
    scalerX=StandardScaler(),
    scalerY=StandardScaler(),
    colsX=colsX,
    colsY=colsY,
)
idx = 1
random_state = 1
test_size = 0.0
data_handler.split_and_scale(random_state=random_state, test_size=test_size)
df_X_train, df_Y_train = data_handler.get_train(as_dataframe=True)
display(df_X_train.head())
display(df_Y_train.head())

# Analyze model summary (all predictors)
X = sm.add_constant(df_X_train)
model = sm.OLS(df_Y_train, X).fit()

# Get and print model summary
model_summary = model.summary()
print(model_summary)

# Get p-values and sort features by significance
df_table = model.summary2().tables[1]
df_table = df_table.sort_values(by="P>|t|", ascending=True)
display(df_table)

ranking = df_table.reset_index()[["index", "P>|t|"]].rename(
    columns={"index": "feature", "P>|t|": "value"}
)
ranking["measure"] = "OLS_p_value"
ranking["rank"] = np.arange(1, len(ranking) + 1)
# Remove constant term from ranking
ranking = ranking[ranking["feature"] != "const"]
display(ranking)

Selected feature columns: ['position', 'R', 'W', 'D']
Selected target columns: ['stress_value_6061']
dfX.shape: (378, 4)
dfY.shape: (378, 1)
_X.shape: (378, 4)
_Y.shape: (378, 1)
No test set, using all data for training.


,position,R,W,D
0,-5.000000e-01,-1.224745,0.000000,-1.224745
1,-4.810966e-16,1.224745,1.224745,0.000000
2,-1.000000e+00,0.000000,-1.224745,-1.224745
3,-5.000000e-01,1.224745,1.224745,1.224745
4,-1.000000e+00,1.224745,0.000000,-1.224745


,stress_value_6061
0,0.181981
1,0.528620
2,-0.347100
3,0.127249
4,1.696247


                            OLS Regression Results                            
Dep. Variable:      stress_value_6061   R-squared:                       0.015
Model:                            OLS   Adj. R-squared:                  0.005
Method:                 Least Squares   F-statistic:                     1.450
Date:                Thu, 26 Feb 2026   Prob (F-statistic):              0.217
Time:                        07:50:45   Log-Likelihood:                -533.44
No. Observations:                 378   AIC:                             1077.
Df Residuals:                     373   BIC:                             1097.
Df Model:                           4                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const      -5.117e-17      0.051  -9.96e-16      1.0

,Coef.,Std.Err.,t,P>|t|,[0.025,0.975]
W,1.170720e-01,0.05138,2.278549e+00,0.023259,0.016041,0.218103
position,-2.798020e-02,0.05138,-5.445728e-01,0.586373,-0.129011,0.073051
D,2.142510e-02,0.05138,4.169923e-01,0.676924,-0.079606,0.122456
R,1.909990e-02,0.05138,3.717374e-01,0.710299,-0.081931,0.120131
const,-5.117434e-17,0.05138,-9.959958e-16,1.000000,-0.101031,0.101031


,feature,value,measure,rank
0,W,0.023259,OLS_p_value,1
1,position,0.586373,OLS_p_value,2
2,D,0.676924,OLS_p_value,3
3,R,0.710299,OLS_p_value,4
